# CHIMERA Calibration Analysis

This notebook demonstrates how to analyze model calibration using the CHIMERA benchmark framework.

## What is Calibration?

A well-calibrated model's confidence scores reflect its actual accuracy. If a model expresses 80% confidence, it should be correct 80% of the time.

## Contents

1. Setup and Imports
2. Running Calibration Evaluation
3. Computing Calibration Metrics
4. Visualizing Calibration
5. Analyzing Results

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CHIMERA imports
from chimera.evaluation import EvaluationPipeline, PipelineConfig
from chimera.generators import CalibrationTaskGenerator
from chimera.metrics import (
    compute_ece,
    compute_mce,
    compute_brier_score,
    create_reliability_diagram,
)

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Imports successful!")

## 2. Running Calibration Evaluation

First, let's run the calibration track on a model.

In [ ]:
# Configure the evaluation
config = PipelineConfig(
    tracks=["calibration"],
    model_provider="gemini",
    model_name="gemini-2.0-flash",
    n_tasks=100,
    seed=42,
    output_dir="results/calibration_analysis",
)

print(f"Configuration:")
print(f"  Model: {config.model_name}")
print(f"  Tasks: {config.n_tasks}")
print(f"  Seed: {config.seed}")

In [ ]:
# Run the evaluation
# NOTE: This requires API credentials to be set

# Uncomment to run actual evaluation:
# pipeline = EvaluationPipeline(config)
# results = pipeline.run()

# For demonstration, we'll use synthetic data
np.random.seed(42)

# Simulate model predictions
n_samples = 100

# Generate synthetic confidence scores and correctness
# Simulating a slightly overconfident model
confidences = np.random.beta(3, 1.5, n_samples)  # Skewed towards high confidence
# Ground truth: model correct based on confidence + noise
correctness = (np.random.random(n_samples) < confidences * 0.85).astype(int)

print(f"Sample predictions generated:")
print(f"  Mean confidence: {confidences.mean():.2%}")
print(f"  Actual accuracy: {correctness.mean():.2%}")
print(f"  Gap (overconfidence): {confidences.mean() - correctness.mean():.2%}")

## 3. Computing Calibration Metrics

CHIMERA provides several metrics to measure calibration quality.

In [ ]:
# Expected Calibration Error (ECE)
# Lower is better (0 = perfectly calibrated)

def compute_ece_manual(confidences, correctness, n_bins=10):
    """Compute Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_confidence = confidences[mask].mean()
            bin_accuracy = correctness[mask].mean()
            bin_weight = mask.sum() / len(confidences)
            ece += bin_weight * abs(bin_accuracy - bin_confidence)
    
    return ece

ece = compute_ece_manual(confidences, correctness)
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"Interpretation: Average gap between confidence and accuracy is {ece:.1%}")

In [ ]:
# Maximum Calibration Error (MCE)
# Lower is better - captures worst-case miscalibration

def compute_mce_manual(confidences, correctness, n_bins=10):
    """Compute Maximum Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    mce = 0.0
    
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_confidence = confidences[mask].mean()
            bin_accuracy = correctness[mask].mean()
            mce = max(mce, abs(bin_accuracy - bin_confidence))
    
    return mce

mce = compute_mce_manual(confidences, correctness)
print(f"Maximum Calibration Error (MCE): {mce:.4f}")
print(f"Interpretation: Worst bin has {mce:.1%} gap between confidence and accuracy")

In [ ]:
# Brier Score
# Lower is better (0 = perfect, 1 = worst)

def compute_brier_manual(confidences, correctness):
    """Compute Brier Score."""
    return np.mean((confidences - correctness) ** 2)

brier = compute_brier_manual(confidences, correctness)
print(f"Brier Score: {brier:.4f}")
print(f"Interpretation: Mean squared error of probability estimates")

In [ ]:
# Summary of all metrics
print("\n" + "="*50)
print("Calibration Metrics Summary")
print("="*50)
print(f"{'Metric':<30} {'Value':>10} {'Quality':>10}")
print("-"*50)
print(f"{'Expected Calibration Error':<30} {ece:>10.4f} {'Good' if ece < 0.05 else 'Fair' if ece < 0.1 else 'Poor':>10}")
print(f"{'Maximum Calibration Error':<30} {mce:>10.4f} {'Good' if mce < 0.1 else 'Fair' if mce < 0.2 else 'Poor':>10}")
print(f"{'Brier Score':<30} {brier:>10.4f} {'Good' if brier < 0.1 else 'Fair' if brier < 0.2 else 'Poor':>10}")

## 4. Visualizing Calibration

The reliability diagram is the standard visualization for calibration.

In [ ]:
def create_reliability_diagram_manual(confidences, correctness, n_bins=10, title="Reliability Diagram"):
    """Create a reliability diagram."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Compute bin statistics
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2
    bin_accuracies = []
    bin_confidences = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_accuracies.append(correctness[mask].mean())
            bin_confidences.append(confidences[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_accuracies.append(0)
            bin_confidences.append(0)
            bin_counts.append(0)
    
    # Left plot: Reliability diagram
    ax1.bar(bin_centers, bin_accuracies, width=0.08, alpha=0.7, 
            color='steelblue', edgecolor='navy', label='Accuracy')
    ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect calibration')
    
    # Show gap (miscalibration)
    for i, (acc, conf, cnt) in enumerate(zip(bin_accuracies, bin_confidences, bin_counts)):
        if cnt > 0:
            gap_color = 'red' if conf > acc else 'green'
            ax1.plot([bin_centers[i], bin_centers[i]], [acc, bin_centers[i]], 
                    color=gap_color, linewidth=2, alpha=0.6)
    
    ax1.set_xlabel('Confidence', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title(title, fontsize=14)
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.legend(loc='upper left')
    ax1.set_aspect('equal')
    
    # Add ECE annotation
    ax1.text(0.05, 0.95, f'ECE: {ece:.4f}', transform=ax1.transAxes,
             fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Right plot: Confidence histogram
    ax2.bar(bin_centers, bin_counts, width=0.08, alpha=0.7,
            color='steelblue', edgecolor='navy')
    ax2.set_xlabel('Confidence', fontsize=12)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title('Confidence Distribution', fontsize=14)
    ax2.set_xlim(0, 1)
    
    plt.tight_layout()
    return fig

fig = create_reliability_diagram_manual(confidences, correctness)
plt.show()

In [ ]:
# Calibration curve with confidence intervals
def create_calibration_curve_with_ci(confidences, correctness, n_bins=10, n_bootstrap=100):
    """Create calibration curve with bootstrap confidence intervals."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2
    
    # Bootstrap for confidence intervals
    bootstrap_accuracies = np.zeros((n_bootstrap, n_bins))
    
    for b in range(n_bootstrap):
        idx = np.random.choice(len(confidences), len(confidences), replace=True)
        boot_conf = confidences[idx]
        boot_corr = correctness[idx]
        
        for i in range(n_bins):
            mask = (boot_conf > bin_boundaries[i]) & (boot_conf <= bin_boundaries[i + 1])
            if mask.sum() > 0:
                bootstrap_accuracies[b, i] = boot_corr[mask].mean()
    
    # Compute statistics
    mean_acc = bootstrap_accuracies.mean(axis=0)
    ci_low = np.percentile(bootstrap_accuracies, 2.5, axis=0)
    ci_high = np.percentile(bootstrap_accuracies, 97.5, axis=0)
    
    # Plot
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect calibration')
    ax.plot(bin_centers, mean_acc, 'o-', color='steelblue', linewidth=2, 
            markersize=8, label='Observed accuracy')
    ax.fill_between(bin_centers, ci_low, ci_high, alpha=0.3, color='steelblue',
                    label='95% CI')
    
    ax.set_xlabel('Confidence', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Calibration Curve with 95% Confidence Intervals', fontsize=14)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig

fig = create_calibration_curve_with_ci(confidences, correctness)
plt.show()

## 5. Analyzing Results

Let's dig deeper into the calibration patterns.

In [ ]:
# Analyze overconfidence vs underconfidence
def analyze_calibration_direction(confidences, correctness):
    """Analyze whether model is over or underconfident."""
    print("Calibration Direction Analysis")
    print("="*50)
    
    # Overall direction
    mean_conf = confidences.mean()
    mean_acc = correctness.mean()
    
    if mean_conf > mean_acc:
        direction = "OVERCONFIDENT"
        gap = mean_conf - mean_acc
    else:
        direction = "UNDERCONFIDENT"
        gap = mean_acc - mean_conf
    
    print(f"Overall: Model is {direction}")
    print(f"  Mean confidence: {mean_conf:.2%}")
    print(f"  Actual accuracy: {mean_acc:.2%}")
    print(f"  Gap: {gap:.2%}")
    
    # Per-confidence-level analysis
    print("\nPer-confidence-level analysis:")
    thresholds = [0.5, 0.7, 0.9]
    for thresh in thresholds:
        mask = confidences >= thresh
        if mask.sum() > 5:
            acc = correctness[mask].mean()
            conf = confidences[mask].mean()
            status = "over" if conf > acc else "under"
            print(f"  Confidence >= {thresh:.0%}: {acc:.1%} accuracy ({status}confident by {abs(conf-acc):.1%})")

analyze_calibration_direction(confidences, correctness)

In [ ]:
# Compare calibration across difficulty levels
# (Simulated data)

np.random.seed(42)

difficulties = np.random.choice(['easy', 'medium', 'hard'], n_samples, p=[0.3, 0.5, 0.2])

# Adjust accuracy based on difficulty
difficulty_multipliers = {'easy': 1.0, 'medium': 0.85, 'hard': 0.7}
adjusted_correctness = np.array([
    1 if np.random.random() < confidences[i] * difficulty_multipliers[d] else 0
    for i, d in enumerate(difficulties)
])

# Compute ECE per difficulty
print("Calibration by Difficulty Level")
print("="*50)

for diff in ['easy', 'medium', 'hard']:
    mask = difficulties == diff
    if mask.sum() > 10:
        ece_diff = compute_ece_manual(confidences[mask], adjusted_correctness[mask])
        acc_diff = adjusted_correctness[mask].mean()
        conf_diff = confidences[mask].mean()
        print(f"\n{diff.upper()}:")
        print(f"  Samples: {mask.sum()}")
        print(f"  ECE: {ece_diff:.4f}")
        print(f"  Mean confidence: {conf_diff:.2%}")
        print(f"  Mean accuracy: {acc_diff:.2%}")

In [ ]:
# Visualize calibration by difficulty
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = {'easy': 'green', 'medium': 'orange', 'hard': 'red'}

for idx, diff in enumerate(['easy', 'medium', 'hard']):
    ax = axes[idx]
    mask = difficulties == diff
    
    if mask.sum() > 10:
        conf_d = confidences[mask]
        corr_d = adjusted_correctness[mask]
        
        # Compute bins
        n_bins = 5
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2
        
        bin_accuracies = []
        for i in range(n_bins):
            bin_mask = (conf_d > bin_boundaries[i]) & (conf_d <= bin_boundaries[i + 1])
            if bin_mask.sum() > 0:
                bin_accuracies.append(corr_d[bin_mask].mean())
            else:
                bin_accuracies.append(0)
        
        ax.bar(bin_centers, bin_accuracies, width=0.15, alpha=0.7,
               color=colors[diff], edgecolor='black')
        ax.plot([0, 1], [0, 1], 'k--', linewidth=2)
        
        ece_d = compute_ece_manual(conf_d, corr_d)
        ax.set_title(f'{diff.upper()} (ECE: {ece_d:.3f})', fontsize=12)
        ax.set_xlabel('Confidence')
        ax.set_ylabel('Accuracy')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect('equal')

plt.suptitle('Calibration by Difficulty Level', fontsize=14)
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:

1. **Running calibration evaluation** using CHIMERA's evaluation pipeline
2. **Computing calibration metrics** (ECE, MCE, Brier Score)
3. **Visualizing calibration** with reliability diagrams
4. **Analyzing calibration patterns** by difficulty level

### Key Insights

- **ECE** measures average calibration error - aim for < 0.05
- **MCE** measures worst-case error - identifies problematic confidence ranges
- **Brier Score** provides a proper scoring rule for probabilistic predictions
- Models often show different calibration patterns across difficulty levels

### Next Steps

- Try different models to compare their calibration
- Analyze calibration across different task categories
- Explore temperature scaling for calibration improvement